In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

load_dotenv()

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why parrots talk")
response

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots. I remember that parrots are known for their ability to mimic human speech. But why exactly do they do that? Maybe it\'s related to their natural behavior. In the wild, parrots live in social groups, so talking might help them communicate with each other. But when they\'re kept as pets, they learn to mimic humans because they associate human speech with positive experiences, like attention or food. \n\nI should also consider the structure of their vocal system. Parrots have a syrinx, which is the vocal organ in birds, and they can control it to produce a wide range of sounds. Their beaks and tongues might help shape the sounds too. But how does this translate to mimicking human speech specifically? Maybe they pick up on the sounds they hear most often. \n\nAnother angle is the social aspect. Parrots are social animals, so they might talk to interact with their h

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """
    Get the weather of a location
    """
    return f"It's Rainy at {location} and it's not preffered to go out."

model_with_tools = model.bind_tools([get_weather])
model_with_tools

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000142991E8080>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000142991E9340>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather of a location', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [4]:
# Invoking using tool
response = model_with_tools.invoke("What is the weather in boston")
print(response)
for tool_call in response.tool_calls:
    print(f"""Name of the tool: {tool_call['name']}
Location: {tool_call['args']['location']}""")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is "location", which should be a string. Since the user mentioned "boston", I should pass that as the location. I\'ll make sure to format the tool call correctly in JSON inside the XML tags. No other functions are available, so this is the only one I need to call.\n', 'tool_calls': [{'id': 'ta06z5wna', 'function': {'arguments': '{"location":"boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 116, 'prompt_tokens': 153, 'total_tokens': 269, 'completion_time': 0.206987893, 'completion_tokens_details': {'reasoning_tokens': 91}, 'prompt_time': 0.007156054, 'prompt_tokens_details': None, 'queue_time': 0.53732203, 'total_time': 0.214143947}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on

In [5]:
# Tool execution
# Step 1: Model generates tool calls
messages = [{"role":"user", "content": "What is the weather in boston"}]
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)

# Step 2: Execute tools and collect results
for tool_call in ai_message.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The current weather in Boston is rainy. It’s not ideal to go out right now—consider staying indoors or preparing an umbrella if you need to head out! 🌧️
